# 07 — Model Training + Hybrid Submission

**Strategy**: DWS lookup covers ~98%+ of test rows. A trained model fills the gaps.

**Pipeline**:
1. Load train/test data
2. Define features (exclude targets, metadata, DWS predictions)
3. Train Random Forest + XGBoost on each target
4. Compare performance, pick best
5. Predict test set
6. Hybrid: DWS where available → model fills gaps
7. Save submission

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# Try importing xgboost
try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost available')
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed — run: pip install xgboost')

# ============================================================
# CONFIGURE PATHS
# ============================================================
TRAIN_PATH = 'data/dws_matched_all_columns.csv'
TEST_PATH  = 'data/dws_test.csv'
OUTPUT_PATH = 'data/submission_hybrid.csv'

print('Setup complete.')

XGBoost available
Setup complete.


## 1. Load Data

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')

# Check DWS coverage on test set
for col in ['Total Alkalinity (DWS)', 'Electrical Conductance (DWS)', 'Dissolved Reactive Phosphorus (DWS)']:
    n = test[col].notna().sum()
    print(f'  {col}: {n}/{len(test)} ({n/len(test)*100:.1f}%)')

Train shape: (9319, 69)
Test shape:  (200, 69)
  Total Alkalinity (DWS): 103/200 (51.5%)
  Electrical Conductance (DWS): 103/200 (51.5%)
  Dissolved Reactive Phosphorus (DWS): 103/200 (51.5%)


## 2. Define Features

Exclude targets, metadata, DWS prediction columns, and non-numeric columns.

In [3]:
# Target columns
TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Columns to EXCLUDE from features
EXCLUDE_COLS = [
    # Targets
    'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus',
    # DWS predictions (used in hybrid, not as model features)
    'Total Alkalinity (DWS)', 'Electrical Conductance (DWS)', 'Dissolved Reactive Phosphorus (DWS)',
    # Metadata / non-feature columns
    'Sample Date', '_merge_terra', '_merge_landsat', 'Impute_Method',
    'geometry', 'STAT_ID', 'Latitude_glorich', 'Longitude_glorich',
    'date', 'dws_station_id', 'dws_date', 'dws_Station',
    'days_diff', 'date_diff_days',
    # dist_km is redundant with dist_m
    'dist_km',
]

# Build feature list: everything not excluded
feature_cols = [c for c in train.columns if c not in EXCLUDE_COLS]

# Drop any non-numeric columns that slipped through
non_numeric = []
for col in feature_cols:
    if train[col].dtype == 'object':
        non_numeric.append(col)

if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    feature_cols = [c for c in feature_cols if c not in non_numeric]

print(f'Feature columns ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  {c}')

# Check features exist in both train and test
missing_in_test = [c for c in feature_cols if c not in test.columns]
if missing_in_test:
    print(f'\nWARNING: Missing in test: {missing_in_test}')
    feature_cols = [c for c in feature_cols if c in test.columns]

print(f'\nFinal feature count: {len(feature_cols)}')

Feature columns (48):
  Latitude
  Longitude
  pet
  nir
  green
  swir16
  swir22
  NDMI
  MNDWI
  sc
  ss
  su
  mt
  va
  vb
  vi
  pa
  pb
  pi
  GLC_Artificial
  GLC_Managed
  GLC_Water
  GLC_Aquatic_Veg
  GLC_PERC_COV
  Popdens_00
  Soil_pH
  SOC
  Soil_wetness
  dist_m
  Alkalinity
  Cl
  DIP
  SO4
  SpecCond25C
  pH
  Alkalinity_reliability
  Cl_reliability
  DIP_reliability
  SO4_reliability
  SpecCond25C_reliability
  pH_reliability
  dws_pH
  dws_Ca
  dws_Mg
  dws_Na
  dws_Cl
  dws_SO4
  dws_P_Tot

Final feature count: 48


In [4]:
# Prepare X and y
X_train = train[feature_cols].copy()
X_test  = test[feature_cols].copy()

# Check missing values
print('Missing values in training features:')
missing = X_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    for col, count in missing.items():
        print(f'  {col}: {count} ({count/len(X_train)*100:.1f}%)')
else:
    print('  None!')

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

Missing values in training features:
  dws_P_Tot: 8610 (92.4%)
  dws_Na: 6604 (70.9%)
  dws_SO4: 5686 (61.0%)
  dws_Mg: 5652 (60.7%)
  dws_Ca: 5632 (60.4%)
  dws_Cl: 5599 (60.1%)
  dws_pH: 5559 (59.7%)
  DIP: 1600 (17.2%)
  Cl: 1600 (17.2%)
  Alkalinity: 1600 (17.2%)
  SO4: 1600 (17.2%)
  Cl_reliability: 1600 (17.2%)
  SpecCond25C_reliability: 1600 (17.2%)
  DIP_reliability: 1600 (17.2%)
  SO4_reliability: 1600 (17.2%)
  SpecCond25C: 1600 (17.2%)
  Alkalinity_reliability: 1600 (17.2%)
  pH: 1536 (16.5%)
  pH_reliability: 1536 (16.5%)

X_train shape: (9319, 48)
X_test shape:  (200, 48)


## 3. Train Models (RF + XGBoost)

Train both models per target, compare 5-fold CV R², pick the best for each target.

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store results
best_models = {}   # {target: fitted model}
best_names = {}    # {target: 'RF' or 'XGB'}
cv_results = {}    # {target: {model_name: (mean, std)}}

for target in TARGETS:
    print(f'\n{"="*70}')
    print(f'TARGET: {target}')
    print(f'{"="*70}')
    
    y_train = train[target].values
    
    # Check for NaN in target
    valid_mask = ~np.isnan(y_train)
    if valid_mask.sum() < len(y_train):
        print(f'  WARNING: {(~valid_mask).sum()} NaN target values — using {valid_mask.sum()} rows')
    
    X_tr = X_train.loc[valid_mask]
    y_tr = y_train[valid_mask]
    
    cv_results[target] = {}
    
    # --- Random Forest ---
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    )
    rf_scores = cross_val_score(rf, X_tr, y_tr, cv=kf, scoring='r2', n_jobs=-1)
    cv_results[target]['RF'] = (rf_scores.mean(), rf_scores.std())
    print(f'\n  Random Forest   — R²: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})')
    print(f'    Per fold: {[f"{s:.4f}" for s in rf_scores]}')
    
    # --- XGBoost ---
    if HAS_XGB:
        xgb_model = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            n_jobs=-1,
            random_state=42,
            verbosity=0
        )
        xgb_scores = cross_val_score(xgb_model, X_tr, y_tr, cv=kf, scoring='r2', n_jobs=-1)
        cv_results[target]['XGB'] = (xgb_scores.mean(), xgb_scores.std())
        print(f'  XGBoost         — R²: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})')
        print(f'    Per fold: {[f"{s:.4f}" for s in xgb_scores]}')
    
    # --- Pick best ---
    best_name = 'RF'
    best_score = cv_results[target]['RF'][0]
    
    if HAS_XGB and cv_results[target]['XGB'][0] > best_score:
        best_name = 'XGB'
        best_score = cv_results[target]['XGB'][0]
    
    print(f'\n  >>> Best: {best_name} (R² = {best_score:.4f})')
    
    # Fit best model on ALL training data
    if best_name == 'RF':
        final_model = RandomForestRegressor(
            n_estimators=200, max_depth=15, min_samples_leaf=5,
            n_jobs=-1, random_state=42
        )
    else:
        final_model = xgb.XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0,
            n_jobs=-1, random_state=42, verbosity=0
        )
    
    final_model.fit(X_tr, y_tr)
    best_models[target] = final_model
    best_names[target] = best_name

print(f'\n{"="*70}')
print('SUMMARY')
print(f'{"="*70}')
for target in TARGETS:
    print(f'  {target}: {best_names[target]} (R² = {cv_results[target][best_names[target]][0]:.4f})')


TARGET: Total Alkalinity

  Random Forest   — R²: 0.8369 (+/- 0.0083)
    Per fold: ['0.8299', '0.8269', '0.8389', '0.8382', '0.8507']
  XGBoost         — R²: 0.8372 (+/- 0.0074)
    Per fold: ['0.8307', '0.8274', '0.8390', '0.8410', '0.8479']

  >>> Best: XGB (R² = 0.8372)

TARGET: Electrical Conductance

  Random Forest   — R²: 0.8494 (+/- 0.0092)
    Per fold: ['0.8506', '0.8353', '0.8533', '0.8450', '0.8630']
  XGBoost         — R²: 0.8454 (+/- 0.0118)
    Per fold: ['0.8440', '0.8252', '0.8525', '0.8442', '0.8608']

  >>> Best: RF (R² = 0.8494)

TARGET: Dissolved Reactive Phosphorus

  Random Forest   — R²: 0.6756 (+/- 0.0099)
    Per fold: ['0.6807', '0.6794', '0.6870', '0.6579', '0.6731']
  XGBoost         — R²: 0.6704 (+/- 0.0185)
    Per fold: ['0.6779', '0.6802', '0.6950', '0.6433', '0.6556']

  >>> Best: RF (R² = 0.6756)

SUMMARY
  Total Alkalinity: XGB (R² = 0.8372)
  Electrical Conductance: RF (R² = 0.8494)
  Dissolved Reactive Phosphorus: RF (R² = 0.6756)


## 4. Generate Test Predictions

In [6]:
# Predict all test rows with the model
model_preds = {}

for target in TARGETS:
    model_preds[target] = best_models[target].predict(X_test)
    print(f'{target} ({best_names[target]}):')
    print(f'  Range: [{model_preds[target].min():.2f}, {model_preds[target].max():.2f}]')
    print(f'  Mean:  {model_preds[target].mean():.2f}')

Total Alkalinity (XGB):
  Range: [20.87, 270.34]
  Mean:  117.50
Electrical Conductance (RF):
  Range: [103.40, 756.94]
  Mean:  424.24
Dissolved Reactive Phosphorus (RF):
  Range: [14.90, 102.64]
  Mean:  29.96


## 5. Hybrid Submission: DWS First, Model Fills Gaps

In [7]:
# Map target names to DWS column names in test
DWS_COL_MAP = {
    'Total Alkalinity': 'Total Alkalinity (DWS)',
    'Electrical Conductance': 'Electrical Conductance (DWS)',
    'Dissolved Reactive Phosphorus': 'Dissolved Reactive Phosphorus (DWS)',
}

# Build submission
submission = test[['Latitude', 'Longitude', 'Sample Date']].copy()

print(f'{"="*70}')
print('HYBRID SUBMISSION')
print(f'{"="*70}\n')

for target in TARGETS:
    dws_col = DWS_COL_MAP[target]
    
    # Start with DWS values
    submission[target] = test[dws_col].values.copy()
    
    # Identify gaps
    dws_missing = submission[target].isna()
    n_dws = (~dws_missing).sum()
    n_gap = dws_missing.sum()
    
    # Fill gaps with model predictions
    submission.loc[dws_missing, target] = model_preds[target][dws_missing]
    
    # Check for any remaining NaN
    n_still_missing = submission[target].isna().sum()
    
    print(f'{target}:')
    print(f'  DWS filled:       {n_dws:,} ({n_dws/len(test)*100:.1f}%)')
    print(f'  Model filled:     {n_gap - n_still_missing:,} ({(n_gap - n_still_missing)/len(test)*100:.1f}%)')
    print(f'  Still missing:    {n_still_missing}')
    print(f'  Final range:      [{submission[target].min():.2f}, {submission[target].max():.2f}]')
    print()

HYBRID SUBMISSION

Total Alkalinity:
  DWS filled:       103 (51.5%)
  Model filled:     97 (48.5%)
  Still missing:    0
  Final range:      [8.95, 514.18]

Electrical Conductance:
  DWS filled:       103 (51.5%)
  Model filled:     97 (48.5%)
  Still missing:    0
  Final range:      [99.10, 5000.00]

Dissolved Reactive Phosphorus:
  DWS filled:       103 (51.5%)
  Model filled:     97 (48.5%)
  Still missing:    0
  Final range:      [5.00, 1156.00]



In [8]:
# Sanity check: compare DWS-only vs model-only ranges
print('Sanity check — value ranges by source:\n')

for target in TARGETS:
    dws_col = DWS_COL_MAP[target]
    dws_vals = test[dws_col].dropna()
    model_vals = pd.Series(model_preds[target])
    
    print(f'{target}:')
    print(f'  DWS    — mean: {dws_vals.mean():.2f}, std: {dws_vals.std():.2f}, range: [{dws_vals.min():.2f}, {dws_vals.max():.2f}]')
    print(f'  Model  — mean: {model_vals.mean():.2f}, std: {model_vals.std():.2f}, range: [{model_vals.min():.2f}, {model_vals.max():.2f}]')
    print()

Sanity check — value ranges by source:

Total Alkalinity:
  DWS    — mean: 131.83, std: 115.08, range: [8.95, 514.18]
  Model  — mean: 117.50, std: 63.20, range: [20.87, 270.34]

Electrical Conductance:
  DWS    — mean: 622.24, std: 758.38, range: [99.10, 5000.00]
  Model  — mean: 424.24, std: 157.40, range: [103.40, 756.94]

Dissolved Reactive Phosphorus:
  DWS    — mean: 46.66, std: 130.86, range: [5.00, 1156.00]
  Model  — mean: 29.96, std: 15.07, range: [14.90, 102.64]



## 6. Save Submission

In [9]:
# ============================================================
# 6. Save Submission — match template exactly
# ============================================================

# Load template
template = pd.read_csv('data/submission_template.csv')  # adjust path
print(f'Template shape: {template.shape}')
print(f'Template columns: {list(template.columns)}')

# Match predictions to template rows by (Latitude, Longitude, Sample Date)
# Parse dates consistently for matching
template['Sample Date'] = pd.to_datetime(template['Sample Date'], dayfirst=True)
test['Sample Date'] = pd.to_datetime(test['Sample Date'], dayfirst=True)

# Merge our predictions onto the template
submission = submission.copy()
submission['Sample Date'] = pd.to_datetime(submission['Sample Date'], dayfirst=True)

# Merge on the 3 key columns
template_filled = template[['Latitude', 'Longitude', 'Sample Date']].merge(
    submission[['Latitude', 'Longitude', 'Sample Date'] + TARGETS],
    on=['Latitude', 'Longitude', 'Sample Date'],
    how='left'
)

# Check coverage
print(f'\nTemplate rows matched: {template_filled[TARGETS[0]].notna().sum()} / {len(template_filled)}')
for target in TARGETS:
    n_missing = template_filled[target].isna().sum()
    print(f'  {target}: {n_missing} still missing')

# Format dates back to DD-MM-YYYY to match template
template_filled['Sample Date'] = template_filled['Sample Date'].dt.strftime('%d-%m-%Y')

# Ensure column order matches template exactly
template_filled = template_filled[['Latitude', 'Longitude', 'Sample Date',
                                    'Total Alkalinity', 'Electrical Conductance',
                                    'Dissolved Reactive Phosphorus']]

# Save
template_filled.to_csv(OUTPUT_PATH, index=False)
print(f'\nSaved to: {OUTPUT_PATH}')
print(f'Shape: {template_filled.shape}')
template_filled.head()

Template shape: (200, 6)
Template columns: ['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

Template rows matched: 16 / 200
  Total Alkalinity: 184 still missing
  Electrical Conductance: 184 still missing
  Dissolved Reactive Phosphorus: 184 still missing

Saved to: data/submission_hybrid.csv
Shape: (200, 6)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


## 7. (Optional) Training Set Validation — Simulate Hybrid

See what R² the hybrid approach would give on training data.

In [10]:
print('Training set — Hybrid R² (simulated):\n')

for target in TARGETS:
    y_true = train[target].values
    dws_col = DWS_COL_MAP[target]
    
    # DWS values on training set
    y_dws = train[dws_col].values.copy()
    
    # Model OOF predictions via cross-val
    oof_preds = np.full(len(train), np.nan)
    valid_mask = ~np.isnan(y_true)
    
    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_train.loc[valid_mask])):
        # Map back to original indices
        orig_tr = np.where(valid_mask)[0][tr_idx]
        orig_val = np.where(valid_mask)[0][val_idx]
        
        model = best_models[target].__class__(**best_models[target].get_params())
        model.fit(X_train.iloc[orig_tr], y_true[orig_tr])
        oof_preds[orig_val] = model.predict(X_train.iloc[orig_val])
    
    # Hybrid: DWS first, model fills gaps
    y_hybrid = y_dws.copy()
    dws_missing = np.isnan(y_hybrid)
    y_hybrid[dws_missing] = oof_preds[dws_missing]
    
    # Compute R² for each source
    # DWS-only (where DWS available)
    dws_mask = ~np.isnan(y_dws) & valid_mask
    r2_dws = r2_score(y_true[dws_mask], y_dws[dws_mask])
    
    # Model-only (OOF)
    oof_mask = ~np.isnan(oof_preds) & valid_mask
    r2_model = r2_score(y_true[oof_mask], oof_preds[oof_mask])
    
    # Hybrid
    hybrid_mask = ~np.isnan(y_hybrid) & valid_mask
    r2_hybrid = r2_score(y_true[hybrid_mask], y_hybrid[hybrid_mask])
    
    print(f'{target}:')
    print(f'  DWS lookup R²:  {r2_dws:.4f}  (on {dws_mask.sum()} rows)')
    print(f'  Model OOF R²:   {r2_model:.4f}  (on {oof_mask.sum()} rows)')
    print(f'  Hybrid R²:      {r2_hybrid:.4f}  (on {hybrid_mask.sum()} rows)')
    print(f'  DWS gaps filled: {(dws_missing & oof_mask).sum()}')
    print()

Training set — Hybrid R² (simulated):

Total Alkalinity:
  DWS lookup R²:  0.5503  (on 3663 rows)
  Model OOF R²:   0.8372  (on 9319 rows)
  Hybrid R²:      0.7249  (on 9319 rows)
  DWS gaps filled: 5656

Electrical Conductance:
  DWS lookup R²:  -1.8926  (on 3752 rows)
  Model OOF R²:   0.8497  (on 9319 rows)
  Hybrid R²:      -0.2119  (on 9319 rows)
  DWS gaps filled: 5567

Dissolved Reactive Phosphorus:
  DWS lookup R²:  -99.5108  (on 3748 rows)
  Model OOF R²:   0.6761  (on 9319 rows)
  Hybrid R²:      -40.2016  (on 9319 rows)
  DWS gaps filled: 5571

